In [ ]:
# takes the dataset, extracts the test split
# applies each checkpoints of each texture to the images in the test set
# runs yolov8 on these images and saves the results to a csv for later analysis/evaluation

import os
import re
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from collections import defaultdict
from tqdm import tqdm

#config must be the same as training

DATASET_FOLDER   = 'sample_dataset'
METADATA_CSV     = os.path.join(DATASET_FOLDER, 'metadata.csv')


ALLOWED_DISTANCES = [5, 7]
ALLOWED_PITCHES   = [30]

MODEL_PATH_RENDERER = 'models/k3_100epch_wo_custom_loss_model.h5'
MODEL_PATH_YOLO     = 'yolov8n_saved_model'

IMG_SIZE        = (500, 500)
YOLO_SIZE       = (640, 640)
TEX_RES         = 16
TEX_UPSCALE_RES = 256

CONF_THRESHOLD = 0.25
NMS_IOU        = 0.45

CHECKPOINT_FOLDERS = ['noise_checkpoints'] # , 'natural_checkpoints'
OUTPUT_DIR         = 'evaluation_results'

VEHICLE_CLASS_IDS = [2, 7, 5]   # car, truck, bus

CLASSES = {
    0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus',
    6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant',
    11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat',
    16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear',
    22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag',
    27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard',
    32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove',
    36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle',
    40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl',
    46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli',
    51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake',
    56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table',
    61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard',
    67: 'cell phone', 68: 'microwave', 69: 'oven', 70: 'toaster', 71: 'sink',
    72: 'refrigerator', 73: 'book', 74: 'clock', 75: 'vase', 76: 'scissors',
    77: 'teddy bear', 78: 'hair drier', 79: 'toothbrush'
}

# load models

print("Loading models...")
renderer_model = tf.keras.models.load_model(MODEL_PATH_RENDERER, compile=False)

yolo_loaded = tf.saved_model.load(MODEL_PATH_YOLO)
yolo_infer  = yolo_loaded.signatures['serving_default']

os.makedirs(OUTPUT_DIR, exist_ok=True)

# get rows that match the training filters

df_meta = pd.read_csv(METADATA_CSV)

if ALLOWED_DISTANCES:
    df_meta = df_meta[df_meta['distance'].isin(ALLOWED_DISTANCES)]
if ALLOWED_PITCHES:
    df_meta = df_meta[df_meta['pitch'].isin(ALLOWED_PITCHES)]

print(f"Rows after filtering: {len(df_meta)}")

all_valid_rows = []
for _, row in df_meta.iterrows():
    base_name    = str(row['filename']).zfill(5)
    ref_path     = os.path.join(DATASET_FOLDER, 'reference',  f"{base_name}.png")
    mask_path    = os.path.join(DATASET_FOLDER, 'masks',      f"{base_name}.png")
    overlay_path = os.path.join(DATASET_FOLDER, 'overlays',   f"{base_name}.png")

    if os.path.exists(ref_path) and os.path.exists(mask_path) and os.path.exists(overlay_path):
        all_valid_rows.append({
            'filename':   f"{base_name}.png",
            'distance':   np.float32(row['distance']),
            'pitch':      np.float32(row['pitch']),
            'yaw':        np.float32(row['yaw']),
            'vehicle_id': row['vehicle_id'],
        })

print(f"Total valid samples in dataset: {len(all_valid_rows)}")


# get test split in the same way training extracts splits

test_vehicle_buckets = defaultdict(list)
for i, r in enumerate(all_valid_rows):
    test_vehicle_buckets[r['vehicle_id']].append(i)

rng = np.random.default_rng(seed=42)

train_indices, val_indices, test_indices = [], [], []

for vehicle_id, indices in sorted(test_vehicle_buckets.items()):
    indices = np.array(indices)
    rng.shuffle(indices)
    
    n_total = len(indices)
    n_test = max(1, round(n_total * 0.10)) # 10% for eval
    n_val  = max(1, round(n_total * 0.20)) # 20% for validation
    
    test_indices.extend(indices[:n_test].tolist())
    val_indices.extend(indices[n_test : n_test + n_val].tolist())
    train_indices.extend(indices[n_test + n_val:].tolist())

rng.shuffle(test_indices) # shuffle for good measure

test_rows = [all_valid_rows[i] for i in test_indices]

print(f"Stratified test split: {len(test_rows)} samples across {len(test_vehicle_buckets)} vehicle type(s)")

for vid, idxs in sorted(test_vehicle_buckets.items()):
    n_taken = sum(1 for i in test_indices if all_valid_rows[i]['vehicle_id'] == vid)
    print(f"  {vid}: {n_taken} test samples (pool had {len(idxs)})")

# helper functions
def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.image.decode_png(raw, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return tf.image.resize(img, IMG_SIZE)


def load_texture_from_png(png_path):
    img = cv2.imread(png_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = cv2.resize(img, (TEX_RES, TEX_RES), interpolation=cv2.INTER_LINEAR)
    return tf.constant(img, dtype=tf.float32)


def calc_transforms(pitch, yaw):
    yaw_diff   = ((yaw + 45.0) % 90.0) - 45.0
    cond_front = tf.logical_and(yaw > 145.0, yaw < 215.0)
    p = tf.where(cond_front, -pitch, pitch)
    y = tf.where(cond_front, -yaw_diff, 0.0)
    r = tf.where(cond_front, 0.0, -yaw_diff)
    p = tf.where(pitch < 15.0, pitch,     p)
    y = tf.where(pitch < 15.0, 0.0,       y)
    r = tf.where(pitch < 15.0, -yaw_diff, r)
    p = tf.where(pitch > 35.0, -pitch,    p)
    y = tf.where(pitch > 35.0, -yaw_diff, y)
    r = tf.where(pitch > 35.0, 0.0,       r)
    return p, y, r


def get_rotation_matrix(pitch, yaw, roll):
    s = np.pi / 180.0
    p, y, r = pitch * s, yaw * s, roll * s
    Rp = tf.stack([[1., 0., 0.],
                   [0.,  tf.cos(p), -tf.sin(p)],
                   [0.,  tf.sin(p),  tf.cos(p)]])
    Rr = tf.stack([[ tf.cos(r), 0., tf.sin(r)],
                   [0., 1., 0.],
                   [-tf.sin(r), 0., tf.cos(r)]])
    Ry = tf.stack([[tf.cos(y), -tf.sin(y), 0.],
                   [tf.sin(y),  tf.cos(y), 0.],
                   [0., 0., 1.]])
    return tf.matmul(Rp, tf.matmul(Rr, Ry))


@tf.function
def render_texture(texture, pitch, yaw, roll, distance,
                   uv_scale=100.0, shift_u=0.0, shift_v=0.0):
    out_h, out_w = IMG_SIZE
    f            = 500.0
    high_res_tex = tf.image.resize(texture, [TEX_UPSCALE_RES, TEX_UPSCALE_RES],
                                   method='nearest')
    R            = get_rotation_matrix(pitch, yaw, roll)
    plane_normal = R[:, 2]
    plane_point  = plane_normal * (distance * 100.0)

    grid_x, grid_y = tf.meshgrid(tf.range(out_w), tf.range(out_h))
    ray_dir = tf.stack([
        tf.cast(grid_x, tf.float32) - out_w / 2.0,
        tf.cast(grid_y, tf.float32) - out_h / 2.0,
        tf.ones([out_h, out_w]) * f,
    ], axis=-1)
    camera_pos = tf.constant([0.0, 0.0, -f])

    denom = tf.tensordot(ray_dir, plane_normal, axes=1)
    denom = tf.where(tf.abs(denom) < 1e-5, 1e-5, denom)
    t     = tf.tensordot(plane_point - camera_pos, plane_normal, axes=1) / denom

    hit_point = camera_pos + ray_dir * tf.expand_dims(t, -1)
    p_local   = tf.reshape(
        tf.matmul(tf.reshape(hit_point - plane_point, [-1, 3]), R),
        [out_h, out_w, 3],
    )

    u  = tf.math.floormod(p_local[:, :, 0] / uv_scale + shift_u, 1.0)
    v  = tf.math.floormod(p_local[:, :, 1] / uv_scale + shift_v, 1.0)

    tex_h = tf.cast(tf.shape(high_res_tex)[0], tf.float32)
    tex_w = tf.cast(tf.shape(high_res_tex)[1], tf.float32)
    x     = tf.clip_by_value(u * (tex_w - 1.0), 0.0, tex_w - 1.001)
    y_    = tf.clip_by_value(v * (tex_h - 1.0), 0.0, tex_h - 1.001)

    x0 = tf.cast(tf.floor(x),  tf.int32);  x1 = x0 + 1
    y0 = tf.cast(tf.floor(y_), tf.int32);  y1 = y0 + 1

    Ia = tf.gather_nd(high_res_tex, tf.stack([y0, x0], axis=-1))
    Ib = tf.gather_nd(high_res_tex, tf.stack([y1, x0], axis=-1))
    Ic = tf.gather_nd(high_res_tex, tf.stack([y0, x1], axis=-1))
    Id = tf.gather_nd(high_res_tex, tf.stack([y1, x1], axis=-1))

    wa = tf.expand_dims((tf.cast(x1, tf.float32) - x)  * (tf.cast(y1, tf.float32) - y_), -1)
    wb = tf.expand_dims((tf.cast(x1, tf.float32) - x)  * (y_ - tf.cast(y0, tf.float32)), -1)
    wc = tf.expand_dims((x - tf.cast(x0, tf.float32))  * (tf.cast(y1, tf.float32) - y_), -1)
    wd = tf.expand_dims((x - tf.cast(x0, tf.float32))  * (y_ - tf.cast(y0, tf.float32)), -1)
    output = tf.add_n([wa * Ia, wb * Ib, wc * Ic, wd * Id])

    return tf.where(tf.expand_dims(t > 0.0, -1), output, tf.zeros_like(output))


def run_yolo(image_tensor):
    yolo_in = tf.cast(
        tf.image.resize(tf.expand_dims(image_tensor, 0), YOLO_SIZE), tf.float32
    )
    return yolo_infer(images=yolo_in)['output_0']


def max_vehicle_conf(yolo_out):
    all_class_scores = yolo_out[0, 4:, :]
    vehicle_scores   = tf.reduce_max(
        tf.gather(all_class_scores, VEHICLE_CLASS_IDS, axis=0), axis=0
    )
    return float(tf.reduce_max(vehicle_scores).numpy())


def is_vehicle_detected_nms(yolo_out):
    output = np.transpose(yolo_out.numpy()[0])
    scale  = [IMG_SIZE[1] / YOLO_SIZE[1], IMG_SIZE[0] / YOLO_SIZE[0]] * 2

    boxes, confs = [], []
    for row in output:
        cls_id = np.argmax(row[4:])
        score  = row[4:][cls_id]
        if score > CONF_THRESHOLD and cls_id in VEHICLE_CLASS_IDS:
            cx, cy, w, h = row[:4] * scale
            boxes.append([int(cx - w / 2), int(cy - h / 2), int(w), int(h)])
            confs.append(float(score))

    if not boxes:
        return False
    indices = cv2.dnn.NMSBoxes(boxes, confs, CONF_THRESHOLD, NMS_IOU)
    return len(indices) > 0


def apply_patch_to_sample(texture, ref_img, mask_img, overlay_img,
                           distance, pitch, yaw):
    p, y, r    = calc_transforms(pitch, yaw)
    rendered   = render_texture(texture, p, y, r, distance,
                                shift_u=0.0, shift_v=0.0)
    tex_masked = tf.where(
        tf.reduce_max(mask_img, axis=-1, keepdims=True) > 0.1,
        rendered, tf.zeros_like(rendered),
    )
    ref_f32        = tf.cast(ref_img, tf.float32)
    tex_masked_f32 = tf.cast(tex_masked, tf.float32)
    renderer_out   = renderer_model(
        [tf.expand_dims(ref_f32, 0), tf.expand_dims(tex_masked_f32, 0)],
        training=False,
    )[0]
    return tf.where(
        tf.reduce_max(overlay_img, axis=-1, keepdims=True) > 0.05,
        ref_img, renderer_out,
    )



def discover_checkpoints(folder):
    if not os.path.isdir(folder):
        print(f"'{folder}' not found")
        return []
    pattern = re.compile(r'patch_epoch_(\d+)\.png', re.IGNORECASE)
    entries = []
    for fname in os.listdir(folder):
        m = pattern.match(fname)
        if m:
            entries.append((int(m.group(1)), os.path.join(folder, fname)))
    entries.sort(key=lambda x: x[0])
    return entries


# pre-compute baselines

print(f"\nComputing baselines for {len(test_rows)} test samples...")

baselines = []
for row in tqdm(test_rows, desc="Baseline"):
    ref_img  = load_image(os.path.join(DATASET_FOLDER, 'reference', row['filename']))
    yolo_out = run_yolo(tf.cast(ref_img, tf.float32))
    baselines.append({
        'filename':          row['filename'],
        'vehicle_id':        row['vehicle_id'],
        'distance':          float(row['distance']),
        'pitch':             float(row['pitch']),
        'yaw':               float(row['yaw']),
        'baseline_conf':     round(max_vehicle_conf(yolo_out), 6),
        'baseline_detected': int(is_vehicle_detected_nms(yolo_out)),
    })

n_detected = sum(b['baseline_detected'] for b in baselines)
print(f"Baseline done — {n_detected}/{len(baselines)} detected at conf > {CONF_THRESHOLD}")


# main eval loop

for ckpt_folder in CHECKPOINT_FOLDERS:
    checkpoints = [c for c in discover_checkpoints(ckpt_folder) if c[0] % 2 == 0]

    if not checkpoints:
        continue

    print(f"\n{'='*60}")
    print(f"Folder: {ckpt_folder}  |  {len(checkpoints)} checkpoint(s)")
    print(f"{'='*60}")

    all_rows = []

    for epoch_num, ckpt_path in checkpoints:
        texture    = load_texture_from_png(ckpt_path)
        epoch_rows = []

        for i, row in enumerate(tqdm(test_rows,
                                     desc=f"  epoch {epoch_num:>3d}",
                                     leave=False)):
            ref_img     = load_image(os.path.join(DATASET_FOLDER, 'reference',  row['filename']))
            mask_img    = load_image(os.path.join(DATASET_FOLDER, 'masks',      row['filename']))
            overlay_img = load_image(os.path.join(DATASET_FOLDER, 'overlays',   row['filename']))

            adv_img  = apply_patch_to_sample(
                texture, ref_img, mask_img, overlay_img,
                tf.constant(row['distance'], dtype=tf.float32),
                tf.constant(row['pitch'],    dtype=tf.float32),
                tf.constant(row['yaw'],      dtype=tf.float32),
            )
            yolo_out   = run_yolo(tf.cast(adv_img, tf.float32))
            p_conf     = round(max_vehicle_conf(yolo_out), 6)
            p_detected = int(is_vehicle_detected_nms(yolo_out))

            base = baselines[i]
            epoch_rows.append({
                'folder':            ckpt_folder,
                'epoch':             epoch_num,
                'filename':          row['filename'],
                'vehicle_id':        row['vehicle_id'],
                'distance':          float(row['distance']),
                'pitch':             float(row['pitch']),
                'yaw':               float(row['yaw']),
                'baseline_conf':     base['baseline_conf'],
                'baseline_detected': base['baseline_detected'],
                'patched_conf':      p_conf,
                'patched_detected':  p_detected,
                'conf_drop':         round(base['baseline_conf'] - p_conf, 6),
                # evasion: was detected cleanly, no longer detected under attack
                'evasion_success':   int(base['baseline_detected'] == 1
                                         and p_detected == 0),
            })

        n_evasions = sum(r['evasion_success'] for r in epoch_rows)
        n_eligible  = sum(r['baseline_detected'] for r in epoch_rows)
        mean_drop   = np.mean([r['conf_drop'] for r in epoch_rows])
        rate        = (n_evasions / n_eligible * 100) if n_eligible > 0 else 0.0
        print(f"  epoch {epoch_num:>3d} | evasions {n_evasions}/{n_eligible} ({rate:.1f}%) | mean conf drop {mean_drop:.4f}")

        all_rows.extend(epoch_rows)

    out_csv = os.path.join(OUTPUT_DIR, f"{ckpt_folder}_evaluation.csv")
    pd.DataFrame(all_rows).to_csv(out_csv, index=False)
    print(f"{len(all_rows)} rows saved to {out_csv}")

print(f"Evaluation Complete")

Loading models...
Rows after filtering: 5394
Total valid samples in dataset: 5394
Stratified test split: 540 samples across 5 vehicle type(s)
  vehicle.audi.tt: 108 test samples (pool had 1079)
  vehicle.nissan.patrol: 108 test samples (pool had 1079)
  vehicle.seat.leon: 108 test samples (pool had 1079)
  vehicle.tesla.model3: 108 test samples (pool had 1079)
  vehicle.toyota.prius: 108 test samples (pool had 1078)

Computing baselines for 540 test samples...


Baseline: 100%|██████████| 540/540 [00:21<00:00, 24.75it/s]


Baseline done — 531/540 detected at conf > 0.25

Folder: noise_checkpoints  |  9 checkpoint(s)


  epoch   0 | evasions 82/531 (15.4%) | mean conf drop 0.1966


  epoch   2 | evasions 415/531 (78.2%) | mean conf drop 0.6292


...

  epoch   8 | evasions 421/531 (79.3%) | mean conf drop 0.6321


  epoch  10 | evasions 425/531 (80.0%) | mean conf drop 0.6331


  epoch  12 | evasions 432/531 (81.4%) | mean conf drop 0.6381


  epoch  14 | evasions 435/531 (81.9%) | mean conf drop 0.6408


  epoch  16 | evasions 436/531 (82.1%) | mean conf drop 0.6416

  → 4860 rows saved to evaluation_results/noise_checkpoints_evaluation.csv

✓ Evaluation complete. Results in: evaluation_results/
